# 03. VLA 실습 준비 확인

**이 단원의 목적: 현재 열린 환경에서 다음 VLA 실습을 시작할 준비가 되었는지 확인합니다.**

[01](01_vla_theory.ipynb)에서는 VLA의 원리를, [02](02_simulation_stack.ipynb)에서는 실습 도구의 역할을 배웠습니다. 이제 실제 가상 장면을 열기 전에 **Python·GPU·배포 자료가 준비되어 있는지** 확인합니다.

아래 점검 셀 두 개를 위에서부터 실행하고 출력의 뜻을 읽습니다. 새 함수를 구현하거나 `notes` 사전을 작성하는 과제는 없습니다. 점검이 끝나면 [04 환경 관찰](04_environment.ipynb)로 넘어갑니다.

**배정 시간 45분:** 환경과 저장 위치 이해 15분 → 점검 실행·출력 해석 15분 → 결과 확인·필요한 오류 대응 15분. 정상인 경우 의미를 확인한 뒤 다음 단원으로 진행합니다.

## MuJoCo 화면 보기: 로컬·원격 공통 noVNC

JupyterLab과 **개인 시뮬레이션 화면을 브라우저 탭 두 개로 열어둡니다.** [README의 최초 접속 안내](README.md#novnc-접속)를 따르세요. 기본 배정은 사용자 A: Jupyter 8890 / 화면 8990, 사용자 B: Jupyter 8891 / 화면 8991입니다. 본인 배정 포트를 사용하며 원격은 두 포트의 SSH 터널을 유지합니다. noVNC에서 Connect를 누르고 암호 `nvidia`를 입력하세요. 처음에는 빈 바탕 화면이 정상이며 실험을 실행하면 MuJoCo 창이 나타납니다.

03에서는 준비 검사만 수행하므로 MuJoCo 창이 아직 없습니다. noVNC 바탕 화면에 접속할 수 있는지 확인하고 04에서 초기 환경을 엽니다. 컨테이너 생성은 호스트에서 이미 완료한 단계입니다.


## 1. 지금 어디에서 확인하는 건가요?

<!-- procedure:lesson-39-f0d92c:start -->
![열린 컨테이너에서 준비 검사](../교재그림/절차/lesson-39.svg)

03은 이미 열린 컨테이너의 점검입니다. 파일·GPU 검사 통과와 실제 렌더링 성공은 다른 단계입니다.
<!-- procedure:lesson-39-f0d92c:end -->


이 노트북은 **이미 열린 5일차 VLA 컨테이너의 JupyterLab** 에서 읽고 실행합니다. 컨테이너를 새로 만드는 명령은 여기서 실행하지 않습니다. 접속·복구가 필요하면 [5일차 실행 안내](README.md#환경과-실행)를 사용합니다.

| 구성 | 제공하는 것 | 이번에 확인할 내용 |
|---|---|---|
| 전용 이미지·컨테이너 | Python과 VLA 실습 라이브러리 | 준비된 Python 환경을 사용하는가 |
| Jetson GPU | 정책 추론에 필요한 CUDA 계산 장치 | 현재 Python에서 GPU를 사용할 수 있는가 |
| `/workspace` | 호스트에서 연결한 교재 폴더 | 실제 교재와 점검 스크립트가 보이는가 |
| `/workspace/Data` | 배포 모델·캐시·환경 자산, 이후 실습 결과 | 필요한 자료가 연결되어 있는가 |

컨테이너는 실행 환경이고, 모델·교재·결과 파일은 연결한 호스트 폴더에 있습니다. 컨테이너 안의 `/workspace`라는 이름이 같아도 **연결한 호스트 개인 폴더가 다르면 내용은 다를 수 있습니다.**

5일차 가상 카메라는 EGL 방식으로 장면을 그립니다. 이 점검에는 USB 카메라나 공용 Ollama가 필요하지 않습니다. 실제 영상 생성은 다음 04에서 확인합니다.

## 2. 첫 번째 점검 — Python과 자료 위치

아래 셀은 경로를 찾아 **현재 값만 출력** 합니다. 결과 폴더를 만들거나 모델을 실행하지 않습니다.

- **Python:** 보통 `/opt/course/bin/python` 또는 같은 환경의 `python3` 경로가 나옵니다.
- **교재:** 컨테이너에서는 `/workspace`가 기준입니다.
- **Data·모델:** 다음 점검에서 읽을 실제 자료 위치입니다.
- **이후 결과 저장 기준:** 아직 결과가 생겼다는 뜻이 아니라, 다음 실습에서 사용할 위치입니다.

Python 경로가 다른 환경이라면 JupyterLab의 커널 선택에서 강사가 준비한 VLA용 Python 커널을 선택하고 이 셀부터 다시 실행하세요.

In [ ]:
from pathlib import Path
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / '10.VLA/scripts/preflight.py').is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('교재 위치를 찾지 못했습니다. /workspace/10.VLA의 노트북을 열었는지 확인하세요.')

SCRIPTS = ROOT / '10.VLA/scripts'
sys.path.insert(0, str(SCRIPTS))
from course_runtime import DATA, MODEL

print('Python:', sys.executable)
print('교재:', ROOT)
print('Data:', DATA)
print('모델:', MODEL)
print('이후 결과 저장 기준:', DATA / 'logs/day5')

## 3. 두 번째 점검 — 실행 전 검사(preflight)

**preflight** 는 본 실행 전에 준비 상태를 살펴보는 검사입니다. 아래 셀은 제공된 [점검 스크립트](scripts/preflight.py)를 **현재 노트북과 같은 Python** 으로 실행합니다. 새 패키지나 모델을 다운로드하지 않습니다.

| 검사하는 것 | 확인하는 이유 |
|---|---|
| 모델 manifest의 저장소·revision과 파일 크기 | 수업용 체크포인트가 지정된 버전이고 필요한 파일이 있는지 확인 |
| Hugging Face·LIBERO 캐시 폴더와 `config.yaml` | 배포 자료의 기본 경로와 설정이 준비되었는지 확인 |
| CUDA 사용 가능 여부 | 현재 환경에서 NVIDIA GPU를 사용할 수 있는지 확인 |
| 주요 패키지의 설치 버전 조회 | 어떤 실행 라이브러리 버전이 설치되어 있는지 확인 |

**여기서 하지 않는 검사도 있습니다.** 실제 가상 영상 생성, 정책 추론, 집기 성공, 전체 캐시 파일의 완전성은 확인하지 않습니다. 기본 실행은 모델 파일의 SHA-256 해시를 전부 계산하지도 않습니다. 라이브러리 버전을 출력하는 것이 모든 조합의 호환성을 증명하는 것은 아닙니다.

아래 셀이 오류 없이 끝나면 기본 준비 검사를 통과한 것입니다. 출력 없이 오래 기다리거나 오류가 나면 반복 실행하지 말고 아래 오류 해석을 먼저 읽으세요.

In [ ]:
import subprocess

subprocess.run([sys.executable, str(SCRIPTS / 'preflight.py')], check=True)
print('기본 준비 검사 통과 — 출력 항목을 확인한 뒤 04 환경 관찰로 진행하세요.')

## 4. 출력은 이렇게 읽습니다

성공하면 `revision`, `device`, `packages`, `offline`, `note`가 포함된 결과가 표시됩니다. 값 자체를 외우거나 별도 사전에 복사할 필요는 없습니다. **각 값이 무엇을 확인해 주는지 설명할 수 있으면 됩니다.**

| 출력 항목 | 의미 | 읽는 방법 |
|---|---|---|
| `revision` | 준비된 모델 스냅샷의 식별자 | 현재 교재의 지정값 `31d453f7edd78c839a8bbc39744a292686daf0de`와 일치해야 함. 스크립트가 manifest와 대조 |
| `device` | 현재 Python에서 사용하는 CUDA 장치 이름 | 실제 장비가 보고한 이름을 읽음. 문구를 특정 GPU 이름으로 바꾸어 적지 않음 |
| `packages` | torch·lerobot·mujoco·hf-libero·robosuite의 설치 버전 | 오류 대응 시 현재 조합을 확인할 근거. 임의로 업데이트하라는 안내가 아님 |
| `offline` | Hugging Face 오프라인 설정 값 | 문자열 `"1"`이면 해당 오프라인 모드가 설정됨. 교재 코드가 이 값을 설정함 |
| `note` | 이 점검으로 확인하지 못한 범위 | 실제 환경 관찰·추론·반복 실행에서 추가 확인해야 함 |

**offline=1은 네트워크가 물리적으로 차단됐다는 뜻이 아닙니다.** 모든 자료가 빠짐없이 존재하거나 모든 프로그램의 인터넷 접근이 막혔다는 증거도 아닙니다.

`CompletedProcess(..., returncode=0)` 같은 문구가 추가로 보이면 점검 프로세스가 정상 종료됐다는 의미입니다. **점검 통과와 VLA의 작업 성공은 다릅니다.** 아직 로봇을 움직이지 않았습니다.

## 5. 오류가 나면 어디를 확인할까요?

아래는 오류에 따라 먼저 확인할 위치입니다. 점검을 통과시키려고 오류를 숨기거나 출력 값을 정답처럼 바꾸지 않습니다.

| 보이는 오류·현상 | 먼저 확인할 것 | 다시 확인하는 순서 |
|---|---|---|
| 교재 위치를 찾지 못함 | `/workspace/10.VLA`의 노트북인지, 교재 폴더가 연결됐는지 | 교재 위치·마운트 확인 후 첫 셀부터 |
| `ModuleNotFoundError` 또는 패키지 버전 조회 오류 | VLA 전용 커널·이미지를 사용하는지 | 커널 확인 → 해결되지 않으면 강사에게 환경 확인 요청 |
| 모델·캐시·config 파일 누락 | 첫 셀의 Data·모델 경로와 연결한 호스트 개인 폴더 | 배포본·마운트 확인 후 점검 셀 재실행 |
| revision 불일치·모델 파일 불완전 | 지정된 모델 묶음이 연결됐는지 | 강사 배포본 확인. manifest 내용만 수정해 우회하지 않음 |
| NVIDIA runtime/CUDA 오류 | 전용 컨테이너에서 GPU가 연결됐는지 | 강사와 런처·GPU 접근 상태 확인 |
| 마지막에 `CalledProcessError` | 그 위에 출력된 구체적인 오류 | 마지막 줄은 점검 실패 요약이므로 위 원인부터 읽기 |

수강생이 임의로 패키지를 설치하거나 모델을 다운로드하는 단계가 아닙니다. 필요한 환경 복구는 [실행·복구 안내](README.md#복구)를 따릅니다. 커널을 재시작했다면 첫 셀부터 다시 실행합니다.

## 6. 결과 파일은 어디에 남나요?

**이 노트북은 점검 결과를 화면에 보여 주며, 별도의 실험 결과 파일을 만들지 않습니다.** 첫 셀에 나온 `Data/logs/day5`는 이후 실습 결과의 저장 기준입니다.

04부터는 `SESSION`이라는 이름으로 실험 회차를 정합니다. 기본 회차의 결과는 `Data/logs/day5/session-01/<실행이름>/`에 저장됩니다. 같은 비교를 이어갈 때 04–08의 회차를 맞추고, 새 비교를 시작할 때 함께 변경합니다.

연결한 호스트 폴더의 파일은 컨테이너를 다시 만들어도 유지할 수 있습니다. 반면 커널 변수와 실행 중 프로세스는 복구되지 않습니다. **저장한 결과 파일과 메모리의 변수를 구분하세요.**

## 7. 다음 실습으로 넘어갈 조건

아래 세 가지를 확인하면 [04 환경 관찰](04_environment.ipynb)로 진행합니다.

1. 현재 Python과 교재·Data 위치를 확인했습니다.
2. preflight가 오류 없이 끝났고, GPU·모델 revision·offline 출력의 의미를 이해했습니다.
3. 이번에 확인한 것은 **기본 준비 상태** 이며, 가상 영상 생성과 정책 추론·작업 성공은 아직 확인하지 않았다는 것을 설명할 수 있습니다.

**이해 확인:** “점검은 통과했는데 다음 단원에서 가상 카메라가 검게 나오면 모순인가요?”

<details>
<summary>해설 보기</summary>

모순이 아닙니다. preflight는 모델 기본 파일·캐시 경로·CUDA·설치 버전을 확인하지만, 실제 EGL 렌더링으로 장면을 만들지는 않습니다. 다음 04에서 실제 환경과 영상을 확인해야 합니다. 같은 이유로 준비 점검 통과만으로 추론 속도나 작업 성공을 보장하지 않습니다.

</details>

<!-- slide-sync:start -->
**슬라이드 복습:** [5일차 25쪽](https://docs.google.com/presentation/d/1A5eYYMofF0rC7Vmz98XHcc92thRzPhBfkyEKHYWyYYw/edit#slide=id.d5_s021) · [배포 PDF 25쪽](../슬라이드/5일차.pdf#page=25)에서 **25–27쪽** 을 보며 출력과 점검의 범위를 설명합니다. 컨테이너 생성은 이 노트북의 과제가 아닙니다.
<!-- slide-sync:end -->